# 1.挂载Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# 验证文件存在
!ls /content/drive/MyDrive/Code/BIT-Control-Project/

Mounted at /content/drive
configs  Download.ipynb  models  openseed  utils


# 2.安装依赖

In [2]:
# 安装核心依赖
!pip install flask pyngrok -q

# 安装 Detectron2 (预编译版本，避免编译问题)
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

# 根据 PyTorch 版本选择合适的 Detectron2
!pip install 'git+https://github.com/facebookresearch/detectron2.git' -q

# 其他依赖
!pip install timm ftfy regex -q

print("✅ 依赖安装完成")

PyTorch: 2.9.0+cu126, CUDA: 12.6
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00
✅ 依赖安装完成


# 3. 编译 CUDA 扩展 (MultiScaleDeformableAttention)

In [3]:
# 修复 PyTorch 2.x 兼容性问题
cuda_file = "/content/drive/MyDrive/Code/BIT-Control-Project/openseed/body/encoder/ops/src/cuda/ms_deform_attn_cuda.cu"

try:
    with open(cuda_file, 'r') as f:
        content = f.read()

    # 替换废弃的 API
    content = content.replace(
        'AT_DISPATCH_FLOATING_TYPES(value.type(), "ms_deform_attn_forward_cuda"',
        'AT_DISPATCH_FLOATING_TYPES(value.scalar_type(), "ms_deform_attn_forward_cuda"'
    )
    content = content.replace(
        'AT_DISPATCH_FLOATING_TYPES(value.type(), "ms_deform_attn_backward_cuda"',
        'AT_DISPATCH_FLOATING_TYPES(value.scalar_type(), "ms_deform_attn_backward_cuda"'
    )

    with open(cuda_file, 'w') as f:
        f.write(content)
    print("✅ CUDA 代码已修复")
except FileNotFoundError:
    print("⚠️ CUDA 文件未找到，可能路径不正确")

✅ CUDA 代码已修复


In [4]:
# 编译 CUDA 扩展
%cd /content/drive/MyDrive/Code/BIT-Control-Project/openseed/body/encoder/ops

!rm -rf build/ *.egg-info dist/ *.so
!python setup.py build_ext --inplace

%cd /content
print("✅ CUDA 扩展编译完成")

/content/drive/MyDrive/Code/BIT-Control-Project/openseed/body/encoder/ops
running build_ext
W0204 13:57:18.088000 3727 torch/utils/cpp_extension.py:630] Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
W0204 13:57:18.140000 3727 torch/utils/cpp_extension.py:521] The detected CUDA version (12.5) has a minor version mismatch with the version that was used to compile PyTorch (12.6). Most likely this shouldn't be a problem.
W0204 13:57:18.140000 3727 torch/utils/cpp_extension.py:531] There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.5
building 'MultiScaleDeformableAttention' extension
creating build/temp.linux-x86_64-cpython-312/content/drive/MyDrive/Code/BIT-Control-Project/openseed/body/encoder/ops/src/cpu
creating build/temp.linux-x86_64-cpython-312/content/drive/MyDrive/Code/BIT-Control-Project/openseed/body/encoder/ops/src/cuda
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign

# 4.加载OpenSeeD模型

In [5]:
import sys
import os

# === 设置项目路径 ===
PROJECT_ROOT = '/content/drive/MyDrive/Code/BIT-Control-Project'
OPS_PATH = os.path.join(PROJECT_ROOT, 'openseed/body/encoder/ops')

# 按正确顺序添加路径
sys.path.insert(0, OPS_PATH)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'openseed'))
sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print(f'工作目录: {os.getcwd()}')

import torch
import numpy as np
from PIL import Image
from torchvision import transforms

from openseed.BaseModel import BaseModel
from openseed import build_model
from detectron2.data import MetadataCatalog
from detectron2.utils.colormap import random_color

# ✅ 使用正确的配置加载函数
from utils.arguments import load_opt_command

# === 配置路径 ===
WEIGHTS_PATH = '/content/drive/MyDrive/Code/BIT-Control-Project/models/openseed_swinl_pano_sota.pt'  # 相对路径
CATEGORIES_PATH = 'configs/panoptic_categories_nomerge.txt'

def init_openseed():
    print("🔄 正在加载 OpenSeeD...")

    # ✅ 使用正确的配置加载
    opt = load_opt_command()

    # 加载类别
    thing_classes = []
    with open(CATEGORIES_PATH, 'r') as f:
        for line in f:
            thing_classes.append(line.strip())
    stuff_classes = []

    # 构建模型
    model = BaseModel(opt, build_model(opt)).from_pretrained(WEIGHTS_PATH).eval().cuda()

    # 设置 metadata
    thing_colors = [random_color(rgb=True, maximum=255).astype(np.int64).tolist() for _ in range(len(thing_classes))]
    stuff_colors = []
    thing_dataset_id_to_contiguous_id = {x: x for x in range(len(thing_classes))}
    stuff_dataset_id_to_contiguous_id = {x + len(thing_classes): x for x in range(len(stuff_classes))}

    MetadataCatalog.get("demo").set(
        thing_colors=thing_colors,
        thing_classes=thing_classes,
        thing_dataset_id_to_contiguous_id=thing_dataset_id_to_contiguous_id,
        stuff_colors=stuff_colors,
        stuff_classes=stuff_classes,
        stuff_dataset_id_to_contiguous_id=stuff_dataset_id_to_contiguous_id,
    )

    model.model.sem_seg_head.predictor.lang_encoder.get_text_embeddings(thing_classes + stuff_classes, is_eval=False)
    model.model.metadata = MetadataCatalog.get('demo')
    model.model.sem_seg_head.num_classes = len(thing_classes + stuff_classes)

    print(f"✅ OpenSeeD 加载完成！显存: {torch.cuda.memory_allocated()/1024**3:.2f}GB")
    return model, thing_classes

openseed_model, thing_classes = init_openseed()

工作目录: /content/drive/MyDrive/Code/BIT-Control-Project


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/drive/MyDrive/Code/BIT-Control-Project/openseed/body/encoder/encoder_deform.py:356: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)
/content/drive/MyDrive/Code/BIT-Control-Project/openseed/body/decoder/utils/dino_decoder.py:241: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


🔄 正在加载 OpenSeeD...


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
swin_large_patch4_window12_384_22k.pth: 929MB [00:10, 88.3MB/s]                          
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

self.task_switch  {'coco': True, 'o365': True}


✅ OpenSeeD 加载完成！显存: 6.03GB


# 5.启动API服务器

In [6]:
from pyngrok import ngrok

# 设置你的 ngrok authtoken
ngrok.set_auth_token("3977HNYaDyamyiXC8f5TeU2Hybu_7GtP68JjmH6wx9ftHM6Eu")

In [7]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading
import base64
import io

app = Flask(__name__)

# 图像预处理
transform = transforms.Compose([transforms.Resize(512, interpolation=Image.BICUBIC)])

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "healthy", "model": "openseed"})

@app.route('/openseed/predict', methods=['POST'])
def predict():
    try:
        # 1. 解码图像
        if 'image_base64' not in request.json:
            return jsonify({"error": "Missing image_base64"}), 400

        img_data = base64.b64decode(request.json['image_base64'])
        image_ori = Image.open(io.BytesIO(img_data)).convert('RGB')

        width, height = image_ori.size
        image = transform(image_ori)
        image_np = np.asarray(image)
        image_tensor = torch.from_numpy(image_np.copy()).permute(2, 0, 1).cuda()

        # 2. 推理
        with torch.no_grad():
            batch_inputs = [{'image': image_tensor, 'height': height, 'width': width}]
            outputs = openseed_model.forward(batch_inputs)

        # 3. 提取结果
        pano_seg = outputs[-1]['panoptic_seg'][0]
        pano_seg_info = outputs[-1]['panoptic_seg'][1]

        # 4. 格式化返回数据
        instances = []
        for info in pano_seg_info:
            cat_id = info['category_id']
            if cat_id in openseed_model.model.metadata.thing_dataset_id_to_contiguous_id:
                cat_id = openseed_model.model.metadata.thing_dataset_id_to_contiguous_id[cat_id]
            instances.append({
                "id": int(info['id']),
                "category_id": int(cat_id),
                "category_name": openseed_model.model.metadata.thing_classes[cat_id] if cat_id < len(openseed_model.model.metadata.thing_classes) else "unknown",
                "bbox": info['bbox'].cpu().tolist() if 'bbox' in info else None,
            })

        # 返回 mask 和实例信息
        return jsonify({
            "mask": pano_seg.cpu().numpy().tolist(),
            "instances": instances
        })

    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

# 启动 ngrok 隧道
public_url = ngrok.connect(5000)
print("=" * 60)
print(f"📡 OpenSeeD API 服务地址: {public_url}")
print("=" * 60)
print("\n请在本地终端运行:")
print(f"export OPENSEED_API_URL={public_url}")
print("=" * 60)

# 后台运行 Flask
threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()

📡 OpenSeeD API 服务地址: NgrokTunnel: "https://phylogenetic-fertile-tajuana.ngrok-free.dev" -> "http://localhost:5000"

请在本地终端运行:
export OPENSEED_API_URL=NgrokTunnel: "https://phylogenetic-fertile-tajuana.ngrok-free.dev" -> "http://localhost:5000"
